# Trust Region Policy Optimization
## TRPO and PPO: Stable Policy Gradient Methods

This notebook provides a self-contained, from-scratch implementation of **Trust Region Policy Optimization (TRPO)** and **Proximal Policy Optimization (PPO)** for reinforcement learning. We develop the theory behind constrained policy updates, implement both algorithms with conjugate gradient, line search, and clipped surrogate objectives, and apply them to continuous control tasks using PyTorch.

**What you'll learn:**
1. Why unconstrained policy gradients can be unstable (large updates destroying performance)
2. Surrogate objective and importance sampling for policy optimization
3. TRPO: trust region constraint with conjugate gradient and line search
4. PPO: clipped surrogate objective as a simpler alternative
5. KL divergence tracking and its role in stability
6. Continuous control with Pendulum-v1

**Prerequisites:** Policy gradients (Notebook 7), Actor-Critic (Notebook 8), basic optimization (conjugate gradient).

**References:**
- Schulman, J., Levine, S., Abbeel, P., Jordan, M., Moritz, P., *Trust Region Policy Optimization*, ICML, 2015.
- Schulman, J., Wolski, F., Dhariwal, P., Radford, A., Klimov, O., *Proximal Policy Optimization Algorithms*, arXiv:1707.06347, 2017.
- Kakade, S., *A Natural Policy Gradient*, NeurIPS, 2002.

---
## 1. Imports and Configuration

In [ ]:
# ============================================================
#  Imports and Configuration
# ============================================================

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.distributions import Normal
import gymnasium as gym
import numpy as np
import matplotlib.pyplot as plt
from typing import List, Tuple, Dict, Optional
from copy import deepcopy
import warnings
warnings.filterwarnings('ignore')

# ---- reproducibility ----
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

# ---- plotting defaults ----
plt.rcParams.update({
    'figure.figsize': (12, 5),
    'font.size': 12,
    'axes.grid': True,
    'grid.alpha': 0.3,
    'lines.linewidth': 2,
})

# ---- color palette ----
COLORS = {
    'trpo': 'steelblue',
    'ppo': 'coral',
    'reinforce': 'seagreen',
    'highlight': 'goldenrod',
    'extra': 'mediumpurple',
}

# ---- hyperparameters ----
GAMMA = 0.99              # discount factor
GAE_LAMBDA = 0.95         # GAE lambda for advantage estimation
LR = 3e-4                 # learning rate (PPO / value network)
CLIP_EPS = 0.2            # PPO clipping epsilon
KL_TARGET = 0.01          # target KL divergence for monitoring
TRPO_DELTA = 0.01         # TRPO trust region constraint
N_EPOCHS = 10             # PPO epochs per update
BATCH_SIZE = 64           # PPO mini-batch size
N_EPISODES = 500          # training episodes
HIDDEN_DIM = 64           # hidden layer size
STEPS_PER_UPDATE = 2048   # steps collected before each update

# ---- device ----
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")
print(f"PyTorch version: {torch.__version__}")
print(f"Gymnasium version: {gym.__version__}")

---
## 2. The Policy Update Problem

Standard policy gradient methods compute a gradient $\nabla_\theta J(\theta)$ and take a step $\theta \leftarrow \theta + \alpha \nabla_\theta J(\theta)$. The fundamental problem is that **the learning rate $\alpha$ is difficult to set**:

- **Too large**: The policy changes drastically, possibly visiting catastrophically bad states. Because the gradient was computed under the *old* policy, the update direction may no longer be valid for the *new* policy. Performance can collapse irreversibly.
- **Too small**: Training is extremely slow; we waste most of our collected data.

Unlike supervised learning, RL has a critical feedback loop: the policy determines the data distribution. A bad policy update means bad data in the next iteration, causing a **death spiral**.

### Conservative Policy Iteration

Kakade & Langford (2002) showed that we can guarantee **monotonic improvement** if we limit the change between consecutive policies. The key result is the **Performance Difference Lemma**:

$$J(\pi') - J(\pi) = \mathbb{E}_{s \sim d^{\pi'}, a \sim \pi'}\left[A^\pi(s, a)\right]$$

where $d^{\pi'}$ is the state visitation distribution under the *new* policy $\pi'$. The challenge is that we need data from $\pi'$ to evaluate $\pi'$, but we only have data from $\pi$. This is where **importance sampling** and **trust regions** come in.

---
## 3. Surrogate Objective and Importance Sampling

To evaluate the new policy $\pi_\theta$ using data collected from the old policy $\pi_{\theta_{\text{old}}}$, we use the **importance sampling ratio**:

$$r_t(\theta) = \frac{\pi_\theta(a_t|s_t)}{\pi_{\theta_{\text{old}}}(a_t|s_t)}$$

The **surrogate objective** (also called the Conservative Policy Iteration objective) replaces the true objective with one we can estimate from old data:

$$\boxed{L^{CPI}(\theta) = \mathbb{E}_t\left[\frac{\pi_\theta(a_t|s_t)}{\pi_{\theta_{\text{old}}}(a_t|s_t)} \hat{A}_t\right] = \mathbb{E}_t\left[r_t(\theta) \hat{A}_t\right]}$$

where $\hat{A}_t$ is an estimator of the advantage function at timestep $t$.

**Key insight**: At $\theta = \theta_{\text{old}}$, the ratio $r_t = 1$ and $L^{CPI} = \mathbb{E}_t[\hat{A}_t]$. The gradient of $L^{CPI}$ matches the policy gradient at this point. However, maximizing $L^{CPI}$ without constraint can lead to excessively large policy updates because the importance weights $r_t$ can blow up.

---
## 4. TRPO: Trust Region Constraint

TRPO constrains the policy update to a **trust region** defined by the KL divergence between old and new policies:

$$\max_\theta \; L(\theta) = \mathbb{E}_t\left[r_t(\theta) \hat{A}_t\right] \quad \text{s.t.} \quad \overline{D}_{KL}(\pi_{\theta_{\text{old}}} \| \pi_\theta) \leq \delta$$

where $\overline{D}_{KL}$ is the average KL divergence over states.

### Natural Gradient and Fisher Information Matrix

The KL constraint induces a **natural gradient** direction. To second order, the KL divergence is:

$$D_{KL}(\pi_{\theta_{\text{old}}} \| \pi_\theta) \approx \frac{1}{2}(\theta - \theta_{\text{old}})^T F (\theta - \theta_{\text{old}})$$

where $F$ is the **Fisher Information Matrix** (FIM). The constrained optimization becomes:

$$\theta^* = \theta_{\text{old}} + \sqrt{\frac{2\delta}{g^T F^{-1} g}} \; F^{-1} g$$

where $g = \nabla_\theta L(\theta)|_{\theta_{\text{old}}}$ is the policy gradient.

### Conjugate Gradient

Computing $F^{-1}g$ directly is prohibitive for large networks. Instead, we use the **conjugate gradient** algorithm to solve $Fx = g$ iteratively, requiring only matrix-vector products $Fv$ (computed via automatic differentiation).

### Line Search with Backtracking

After computing the search direction $s = F^{-1}g$, we use backtracking line search to find the largest step that:
1. Satisfies the KL constraint: $D_{KL} \leq \delta$
2. Actually improves the surrogate objective: $L(\theta_{\text{new}}) > L(\theta_{\text{old}})$

### TRPO Algorithm

```
for iteration = 1, 2, ...:
    1. Collect trajectories under current policy pi_old
    2. Compute advantages A_hat using GAE
    3. Compute policy gradient g = nabla L(theta)
    4. Compute search direction s = F^{-1} g via conjugate gradient
    5. Compute max step size beta = sqrt(2 delta / (s^T F s))
    6. Line search: theta_new = theta_old + alpha * beta * s
       for alpha in [1, 0.5, 0.25, ...] until KL <= delta and L improves
    7. Update value function by regression on returns
```

---
## 5. PPO: Clipped Surrogate Objective

PPO achieves similar stability to TRPO with a much simpler implementation. Instead of a hard KL constraint, PPO **clips** the importance ratio to prevent large updates:

$$\boxed{L^{CLIP}(\theta) = \mathbb{E}_t\left[\min\left(r_t(\theta)\hat{A}_t, \; \text{clip}(r_t(\theta), 1-\epsilon, 1+\epsilon)\hat{A}_t\right)\right]}$$

### Why Clipping Works

Consider the two cases:

- **Positive advantage** ($\hat{A}_t > 0$): The action was better than average. We want to increase its probability, i.e., increase $r_t$. But clipping at $1 + \epsilon$ prevents $r_t$ from growing too large.

- **Negative advantage** ($\hat{A}_t < 0$): The action was worse than average. We want to decrease its probability, i.e., decrease $r_t$. But clipping at $1 - \epsilon$ prevents $r_t$ from shrinking too much.

The $\min$ ensures we take the **pessimistic** (lower) bound, so we only change the policy when both the clipped and unclipped objectives agree.

### PPO Combined Loss

The total PPO loss combines policy, value, and entropy terms:

$$L(\theta) = -L^{CLIP}(\theta) + c_1 \cdot L^{VF}(\theta) - c_2 \cdot S[\pi_\theta](s_t)$$

where $L^{VF}$ is the value function loss (MSE) and $S$ is the entropy bonus for exploration.

---
## 6. Generalized Advantage Estimation (GAE)

Both TRPO and PPO benefit from **Generalized Advantage Estimation** (Schulman et al., 2016), which provides a smooth trade-off between bias and variance in advantage estimation:

$$\hat{A}_t^{GAE(\gamma, \lambda)} = \sum_{l=0}^{\infty} (\gamma \lambda)^l \delta_{t+l}$$

where $\delta_t = r_t + \gamma V(s_{t+1}) - V(s_t)$ is the TD residual.

- $\lambda = 0$: One-step TD advantage (low variance, high bias)
- $\lambda = 1$: Monte Carlo advantage (high variance, low bias)
- $\lambda \in (0, 1)$: Exponentially-weighted blend

The typical choice $\lambda = 0.95$ works well in practice.

---
## 7. Network Architectures

In [ ]:
# ============================================================
#  Gaussian Policy Network
# ============================================================

class GaussianPolicy(nn.Module):
    """Continuous Gaussian policy with state-independent log_std.
    
    Architecture: state -> FC -> Tanh -> FC -> Tanh -> mean
    log_std is a learnable parameter (not state-dependent) for stability.
    """
    
    def __init__(self, state_dim: int, action_dim: int, hidden_dim: int = HIDDEN_DIM):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(state_dim, hidden_dim),
            nn.Tanh(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.Tanh(),
            nn.Linear(hidden_dim, action_dim),
        )
        # State-independent log standard deviation
        self.log_std = nn.Parameter(torch.zeros(action_dim))
    
    def forward(self, state: torch.Tensor) -> Normal:
        """Return a Normal distribution over actions."""
        mean = self.net(state)
        std = self.log_std.exp().expand_as(mean)
        return Normal(mean, std)
    
    def log_prob(self, state: torch.Tensor, action: torch.Tensor) -> torch.Tensor:
        """Compute log probability of action under current policy."""
        dist = self.forward(state)
        return dist.log_prob(action).sum(dim=-1)
    
    def entropy(self, state: torch.Tensor) -> torch.Tensor:
        """Compute entropy of the policy at given states."""
        dist = self.forward(state)
        return dist.entropy().sum(dim=-1)
    
    def select_action(self, state: np.ndarray) -> Tuple[np.ndarray, float]:
        """Sample action and return (action, log_prob)."""
        state_t = torch.FloatTensor(state).unsqueeze(0).to(device)
        dist = self.forward(state_t)
        action = dist.sample()
        log_prob = dist.log_prob(action).sum(dim=-1)
        return action.cpu().detach().numpy().flatten(), log_prob.item()
    
    def get_mean_std(self, state: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
        """Return mean and std for given states."""
        mean = self.net(state)
        std = self.log_std.exp().expand_as(mean)
        return mean, std


def compute_kl_divergence(policy_old: GaussianPolicy, policy_new: GaussianPolicy,
                          states: torch.Tensor) -> torch.Tensor:
    """Compute mean KL divergence KL(pi_old || pi_new) over a batch of states.
    
    For Gaussian policies: KL(N(mu1,s1) || N(mu2,s2)) =
        log(s2/s1) + (s1^2 + (mu1-mu2)^2)/(2*s2^2) - 1/2
    """
    mu_old, std_old = policy_old.get_mean_std(states)
    mu_new, std_new = policy_new.get_mean_std(states)
    
    kl = (torch.log(std_new / std_old)
          + (std_old.pow(2) + (mu_old - mu_new).pow(2)) / (2.0 * std_new.pow(2))
          - 0.5)
    return kl.sum(dim=-1).mean()


# ============================================================
#  Value Network
# ============================================================

class ValueNetwork(nn.Module):
    """State-value function V(s) for advantage estimation."""
    
    def __init__(self, state_dim: int, hidden_dim: int = HIDDEN_DIM):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(state_dim, hidden_dim),
            nn.Tanh(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.Tanh(),
            nn.Linear(hidden_dim, 1),
        )
    
    def forward(self, state: torch.Tensor) -> torch.Tensor:
        return self.net(state).squeeze(-1)


# Quick sanity check
_gp = GaussianPolicy(3, 1)
_vn = ValueNetwork(3)
print(f"GaussianPolicy params:  {sum(p.numel() for p in _gp.parameters()):,}")
print(f"ValueNetwork params:    {sum(p.numel() for p in _vn.parameters()):,}")
print("Network architectures defined.")

---
## 8. Trajectory Collection and GAE

In [ ]:
# ============================================================
#  Trajectory Collection and GAE Computation
# ============================================================

def collect_trajectories(env: gym.Env, policy: GaussianPolicy, value_net: ValueNetwork,
                         n_steps: int, gamma: float = GAMMA,
                         lam: float = GAE_LAMBDA) -> Dict[str, torch.Tensor]:
    """Collect n_steps of experience and compute GAE advantages.
    
    Returns a dictionary with:
        states, actions, log_probs, returns, advantages,
        values, rewards_per_episode (list of episode total rewards)
    """
    states_list = []
    actions_list = []
    log_probs_list = []
    rewards_list = []
    dones_list = []
    values_list = []
    episode_rewards = []
    
    state, _ = env.reset()
    ep_reward = 0.0
    
    for _ in range(n_steps):
        state_t = torch.FloatTensor(state).unsqueeze(0).to(device)
        
        with torch.no_grad():
            value = value_net(state_t).item()
        
        action, log_prob = policy.select_action(state)
        next_state, reward, terminated, truncated, _ = env.step(action)
        done = terminated or truncated
        
        states_list.append(state)
        actions_list.append(action)
        log_probs_list.append(log_prob)
        rewards_list.append(reward)
        dones_list.append(done)
        values_list.append(value)
        
        ep_reward += reward
        state = next_state
        
        if done:
            episode_rewards.append(ep_reward)
            ep_reward = 0.0
            state, _ = env.reset()
    
    # Bootstrap value for last state
    with torch.no_grad():
        last_val = value_net(torch.FloatTensor(state).unsqueeze(0).to(device)).item()
    
    # ---- Compute GAE advantages and returns ----
    advantages = np.zeros(n_steps, dtype=np.float32)
    returns = np.zeros(n_steps, dtype=np.float32)
    gae = 0.0
    
    for t in reversed(range(n_steps)):
        if t == n_steps - 1:
            next_value = last_val
            next_non_terminal = 1.0 - float(dones_list[t])
        else:
            next_value = values_list[t + 1]
            next_non_terminal = 1.0 - float(dones_list[t])
        
        # TD residual
        delta = rewards_list[t] + gamma * next_value * next_non_terminal - values_list[t]
        # GAE accumulation
        gae = delta + gamma * lam * next_non_terminal * gae
        advantages[t] = gae
        returns[t] = advantages[t] + values_list[t]
    
    # Convert to tensors
    data = {
        'states': torch.FloatTensor(np.array(states_list)).to(device),
        'actions': torch.FloatTensor(np.array(actions_list)).to(device),
        'log_probs': torch.FloatTensor(np.array(log_probs_list)).to(device),
        'returns': torch.FloatTensor(returns).to(device),
        'advantages': torch.FloatTensor(advantages).to(device),
        'values': torch.FloatTensor(np.array(values_list)).to(device),
        'episode_rewards': episode_rewards,
    }
    
    # Normalize advantages
    data['advantages'] = (data['advantages'] - data['advantages'].mean()) / (data['advantages'].std() + 1e-8)
    
    return data


def smooth(data: List[float], window: int = 20) -> np.ndarray:
    """Smooth a 1D signal using a running average."""
    if len(data) < window:
        return np.array(data)
    kernel = np.ones(window) / window
    return np.convolve(data, kernel, mode='valid')


print("Trajectory collection and GAE computation defined.")

---
## 9. TRPO Implementation

In [ ]:
# ============================================================
#  TRPO: Conjugate Gradient and Fisher Vector Product
# ============================================================

def conjugate_gradient(Fvp_fn, b: torch.Tensor, n_steps: int = 10,
                       residual_tol: float = 1e-10) -> torch.Tensor:
    """Solve Fx = b using conjugate gradient.
    
    Fvp_fn: function that computes Fisher-vector product F @ v
    b: the gradient vector g (right-hand side)
    n_steps: maximum CG iterations
    
    Returns approximate solution x ~ F^{-1} b
    """
    x = torch.zeros_like(b)
    r = b.clone()       # residual r = b - Fx (initially F*0 = 0)
    p = b.clone()       # search direction
    r_dot_r = r.dot(r)
    
    for _ in range(n_steps):
        Fp = Fvp_fn(p)
        alpha = r_dot_r / (p.dot(Fp) + 1e-8)
        x += alpha * p
        r -= alpha * Fp
        r_dot_r_new = r.dot(r)
        
        if r_dot_r_new < residual_tol:
            break
        
        beta = r_dot_r_new / (r_dot_r + 1e-8)
        p = r + beta * p
        r_dot_r = r_dot_r_new
    
    return x


def fisher_vector_product(policy: GaussianPolicy, states: torch.Tensor,
                          v: torch.Tensor, damping: float = 0.1) -> torch.Tensor:
    """Compute the Fisher-vector product F @ v efficiently.
    
    Uses the identity: F @ v = nabla_theta (nabla_theta KL . v)
    This avoids explicitly forming the Fisher matrix.
    """
    # Compute KL divergence of policy with itself (at current params)
    mu, std = policy.get_mean_std(states)
    dist = Normal(mu, std)
    dist_detached = Normal(mu.detach(), std.detach())
    
    kl = torch.distributions.kl_divergence(dist_detached, dist).sum(dim=-1).mean()
    
    # First derivative of KL w.r.t. policy parameters
    grads = torch.autograd.grad(kl, policy.parameters(), create_graph=True)
    flat_grad_kl = torch.cat([g.reshape(-1) for g in grads])
    
    # Gradient-vector product
    kl_v = flat_grad_kl.dot(v)
    
    # Second derivative (Hessian-vector product = Fisher-vector product)
    grads_grads = torch.autograd.grad(kl_v, policy.parameters())
    flat_fvp = torch.cat([g.reshape(-1).detach() for g in grads_grads])
    
    # Add damping for numerical stability
    return flat_fvp + damping * v


def get_flat_params(model: nn.Module) -> torch.Tensor:
    """Get all parameters as a single flat vector."""
    return torch.cat([p.data.reshape(-1) for p in model.parameters()])


def set_flat_params(model: nn.Module, flat_params: torch.Tensor):
    """Set model parameters from a flat vector."""
    idx = 0
    for p in model.parameters():
        n = p.numel()
        p.data.copy_(flat_params[idx:idx + n].reshape(p.shape))
        idx += n


def surrogate_objective(policy: GaussianPolicy, states: torch.Tensor,
                        actions: torch.Tensor, advantages: torch.Tensor,
                        old_log_probs: torch.Tensor) -> torch.Tensor:
    """Compute the surrogate objective L(theta)."""
    new_log_probs = policy.log_prob(states, actions)
    ratio = (new_log_probs - old_log_probs).exp()
    return (ratio * advantages).mean()


def trpo_step(policy: GaussianPolicy, states: torch.Tensor,
              actions: torch.Tensor, advantages: torch.Tensor,
              old_log_probs: torch.Tensor, max_kl: float = TRPO_DELTA,
              n_cg_steps: int = 10, line_search_steps: int = 10,
              line_search_decay: float = 0.5) -> Dict[str, float]:
    """Perform a single TRPO policy update.
    
    Steps:
    1. Compute policy gradient g
    2. Compute natural gradient direction s = F^{-1}g via conjugate gradient
    3. Compute max step size from KL constraint
    4. Line search for valid step
    
    Returns dict with step info (kl, surrogate improvement, step size).
    """
    # ---- Step 1: Compute policy gradient ----
    loss = surrogate_objective(policy, states, actions, advantages, old_log_probs)
    grads = torch.autograd.grad(loss, policy.parameters())
    flat_grad = torch.cat([g.reshape(-1) for g in grads]).detach()
    
    # ---- Step 2: Compute natural gradient direction via CG ----
    def Fvp_fn(v):
        return fisher_vector_product(policy, states, v)
    
    step_dir = conjugate_gradient(Fvp_fn, flat_grad, n_steps=n_cg_steps)
    
    # ---- Step 3: Compute max step size ----
    sFs = step_dir.dot(Fvp_fn(step_dir))
    max_step_size = torch.sqrt(2 * max_kl / (sFs + 1e-8))
    full_step = max_step_size * step_dir
    
    # ---- Step 4: Line search ----
    old_params = get_flat_params(policy)
    old_loss = loss.item()
    
    # Create a reference copy for KL computation
    old_policy = deepcopy(policy)
    
    step_taken = False
    final_kl = 0.0
    final_step_frac = 0.0
    
    for i in range(line_search_steps):
        step_frac = line_search_decay ** i
        new_params = old_params + step_frac * full_step
        set_flat_params(policy, new_params)
        
        with torch.no_grad():
            new_loss = surrogate_objective(policy, states, actions, advantages, old_log_probs).item()
            kl = compute_kl_divergence(old_policy, policy, states).item()
        
        # Accept step if it improves objective and satisfies KL constraint
        if new_loss > old_loss and kl <= max_kl * 1.5:  # slight tolerance
            step_taken = True
            final_kl = kl
            final_step_frac = step_frac
            break
    
    if not step_taken:
        # Revert to old parameters
        set_flat_params(policy, old_params)
        final_kl = 0.0
        final_step_frac = 0.0
    
    return {
        'kl': final_kl,
        'surrogate_improvement': (new_loss - old_loss) if step_taken else 0.0,
        'step_fraction': final_step_frac,
        'step_taken': step_taken,
    }


print("TRPO components defined: conjugate_gradient, fisher_vector_product, trpo_step.")

---
## 10. PPO Implementation

In [ ]:
# ============================================================
#  PPO: Clipped Surrogate Objective Update
# ============================================================

def ppo_update(policy: GaussianPolicy, value_net: ValueNetwork,
               policy_optimizer: optim.Optimizer, value_optimizer: optim.Optimizer,
               data: Dict[str, torch.Tensor], clip_eps: float = CLIP_EPS,
               n_epochs: int = N_EPOCHS, batch_size: int = BATCH_SIZE,
               entropy_coef: float = 0.01) -> Dict[str, List[float]]:
    """Perform PPO update with clipped surrogate objective.
    
    Runs n_epochs of mini-batch SGD on the collected trajectories.
    
    Returns dict tracking: policy_losses, value_losses, clip_fractions,
                          kl_divs, ratios, entropies
    """
    states = data['states']
    actions = data['actions']
    old_log_probs = data['log_probs']
    returns = data['returns']
    advantages = data['advantages']
    
    n_samples = states.shape[0]
    
    # Create a frozen copy for KL tracking
    old_policy = deepcopy(policy)
    old_policy.eval()
    
    stats = {
        'policy_losses': [],
        'value_losses': [],
        'clip_fractions': [],
        'kl_divs': [],
        'ratios': [],
        'entropies': [],
    }
    
    for epoch in range(n_epochs):
        # Random permutation for mini-batches
        indices = np.random.permutation(n_samples)
        
        for start in range(0, n_samples, batch_size):
            end = min(start + batch_size, n_samples)
            batch_idx = indices[start:end]
            
            batch_states = states[batch_idx]
            batch_actions = actions[batch_idx]
            batch_old_log_probs = old_log_probs[batch_idx]
            batch_returns = returns[batch_idx]
            batch_advantages = advantages[batch_idx]
            
            # ---- Policy loss (clipped surrogate) ----
            new_log_probs = policy.log_prob(batch_states, batch_actions)
            ratio = (new_log_probs - batch_old_log_probs).exp()
            
            # Clipped and unclipped surrogate
            surr1 = ratio * batch_advantages
            surr2 = torch.clamp(ratio, 1.0 - clip_eps, 1.0 + clip_eps) * batch_advantages
            policy_loss = -torch.min(surr1, surr2).mean()
            
            # Entropy bonus for exploration
            entropy = policy.entropy(batch_states).mean()
            policy_loss -= entropy_coef * entropy
            
            # ---- Value loss ----
            value_pred = value_net(batch_states)
            value_loss = F.mse_loss(value_pred, batch_returns)
            
            # ---- Update policy ----
            policy_optimizer.zero_grad()
            policy_loss.backward()
            nn.utils.clip_grad_norm_(policy.parameters(), max_norm=0.5)
            policy_optimizer.step()
            
            # ---- Update value network ----
            value_optimizer.zero_grad()
            value_loss.backward()
            nn.utils.clip_grad_norm_(value_net.parameters(), max_norm=0.5)
            value_optimizer.step()
            
            # ---- Track statistics ----
            with torch.no_grad():
                clip_fraction = ((ratio - 1.0).abs() > clip_eps).float().mean().item()
                stats['policy_losses'].append(policy_loss.item())
                stats['value_losses'].append(value_loss.item())
                stats['clip_fractions'].append(clip_fraction)
                stats['ratios'].extend(ratio.cpu().numpy().tolist())
                stats['entropies'].append(entropy.item())
        
        # Per-epoch KL divergence
        with torch.no_grad():
            kl = compute_kl_divergence(old_policy, policy, states).item()
            stats['kl_divs'].append(kl)
    
    return stats


print("PPO update function defined.")

---
## 11. Vanilla REINFORCE Baseline (for Comparison)

In [ ]:
# ============================================================
#  Vanilla REINFORCE with Baseline (for comparison)
# ============================================================

def train_reinforce(env_name: str = 'Pendulum-v1', n_iterations: int = 200,
                    steps_per_iter: int = STEPS_PER_UPDATE,
                    gamma: float = GAMMA, lr: float = LR,
                    seed: int = SEED) -> Dict[str, list]:
    """Train vanilla REINFORCE with learned baseline on continuous control."""
    np.random.seed(seed)
    torch.manual_seed(seed)
    
    env = gym.make(env_name)
    state_dim = env.observation_space.shape[0]
    action_dim = env.action_space.shape[0]
    
    policy = GaussianPolicy(state_dim, action_dim).to(device)
    value_net = ValueNetwork(state_dim).to(device)
    policy_optimizer = optim.Adam(policy.parameters(), lr=lr)
    value_optimizer = optim.Adam(value_net.parameters(), lr=lr * 3)
    
    history = {'rewards': [], 'kl_divs': []}
    
    for iteration in range(n_iterations):
        data = collect_trajectories(env, policy, value_net, steps_per_iter, gamma)
        
        if data['episode_rewards']:
            history['rewards'].extend(data['episode_rewards'])
        
        # Save old policy for KL tracking
        old_policy = deepcopy(policy)
        
        # ---- Policy update: standard policy gradient ----
        new_log_probs = policy.log_prob(data['states'], data['actions'])
        policy_loss = -(new_log_probs * data['advantages']).mean()
        
        policy_optimizer.zero_grad()
        policy_loss.backward()
        nn.utils.clip_grad_norm_(policy.parameters(), max_norm=0.5)
        policy_optimizer.step()
        
        # ---- Value update ----
        for _ in range(5):
            value_pred = value_net(data['states'])
            value_loss = F.mse_loss(value_pred, data['returns'])
            value_optimizer.zero_grad()
            value_loss.backward()
            value_optimizer.step()
        
        # Track KL
        with torch.no_grad():
            kl = compute_kl_divergence(old_policy, policy, data['states']).item()
            history['kl_divs'].append(kl)
        
        if (iteration + 1) % 50 == 0:
            recent = history['rewards'][-20:] if len(history['rewards']) >= 20 else history['rewards']
            print(f"  REINFORCE iter {iteration+1}/{n_iterations} | "
                  f"Avg Reward: {np.mean(recent):.1f} | KL: {kl:.4f}")
    
    env.close()
    return history


print("Vanilla REINFORCE baseline defined.")

---
## 12. TRPO Training Loop

In [ ]:
# ============================================================
#  TRPO Training on Pendulum-v1
# ============================================================

def train_trpo(env_name: str = 'Pendulum-v1', n_iterations: int = 200,
               steps_per_iter: int = STEPS_PER_UPDATE,
               gamma: float = GAMMA, lam: float = GAE_LAMBDA,
               max_kl: float = TRPO_DELTA, value_lr: float = LR * 3,
               seed: int = SEED) -> Dict[str, list]:
    """Train TRPO agent on a continuous control environment."""
    np.random.seed(seed)
    torch.manual_seed(seed)
    
    env = gym.make(env_name)
    state_dim = env.observation_space.shape[0]
    action_dim = env.action_space.shape[0]
    
    policy = GaussianPolicy(state_dim, action_dim).to(device)
    value_net = ValueNetwork(state_dim).to(device)
    value_optimizer = optim.Adam(value_net.parameters(), lr=value_lr)
    
    history = {
        'rewards': [],
        'kl_divs': [],
        'surrogate_improvements': [],
        'step_fractions': [],
        'policy_means': [],
        'policy_stds': [],
    }
    
    for iteration in range(n_iterations):
        # ---- Collect trajectories ----
        data = collect_trajectories(env, policy, value_net, steps_per_iter, gamma, lam)
        
        if data['episode_rewards']:
            history['rewards'].extend(data['episode_rewards'])
        
        # ---- TRPO policy update ----
        step_info = trpo_step(
            policy, data['states'], data['actions'],
            data['advantages'], data['log_probs'], max_kl=max_kl
        )
        
        history['kl_divs'].append(step_info['kl'])
        history['surrogate_improvements'].append(step_info['surrogate_improvement'])
        history['step_fractions'].append(step_info['step_fraction'])
        
        # ---- Value function update (multiple gradient steps) ----
        for _ in range(10):
            value_pred = value_net(data['states'])
            value_loss = F.mse_loss(value_pred, data['returns'])
            value_optimizer.zero_grad()
            value_loss.backward()
            value_optimizer.step()
        
        # ---- Track policy statistics ----
        with torch.no_grad():
            test_state = torch.zeros(1, state_dim).to(device)
            mu, std = policy.get_mean_std(test_state)
            history['policy_means'].append(mu.cpu().numpy().flatten()[0])
            history['policy_stds'].append(std.cpu().numpy().flatten()[0])
        
        if (iteration + 1) % 50 == 0:
            recent = history['rewards'][-20:] if len(history['rewards']) >= 20 else history['rewards']
            print(f"  TRPO iter {iteration+1}/{n_iterations} | "
                  f"Avg Reward: {np.mean(recent):.1f} | "
                  f"KL: {step_info['kl']:.4f} | "
                  f"Step frac: {step_info['step_fraction']:.3f}")
    
    env.close()
    return history


print("Training TRPO on Pendulum-v1...")
trpo_history = train_trpo()
print(f"TRPO final avg reward (last 20): {np.mean(trpo_history['rewards'][-20:]):.1f}")

---
## 13. PPO Training Loop

In [ ]:
# ============================================================
#  PPO Training on Pendulum-v1
# ============================================================

def train_ppo(env_name: str = 'Pendulum-v1', n_iterations: int = 200,
              steps_per_iter: int = STEPS_PER_UPDATE,
              gamma: float = GAMMA, lam: float = GAE_LAMBDA,
              clip_eps: float = CLIP_EPS, n_epochs: int = N_EPOCHS,
              batch_size: int = BATCH_SIZE, lr: float = LR,
              seed: int = SEED) -> Dict[str, list]:
    """Train PPO agent on a continuous control environment."""
    np.random.seed(seed)
    torch.manual_seed(seed)
    
    env = gym.make(env_name)
    state_dim = env.observation_space.shape[0]
    action_dim = env.action_space.shape[0]
    
    policy = GaussianPolicy(state_dim, action_dim).to(device)
    value_net = ValueNetwork(state_dim).to(device)
    policy_optimizer = optim.Adam(policy.parameters(), lr=lr)
    value_optimizer = optim.Adam(value_net.parameters(), lr=lr * 3)
    
    history = {
        'rewards': [],
        'kl_divs': [],
        'clip_fractions': [],
        'all_ratios': [],
        'policy_losses': [],
        'value_losses': [],
        'entropies': [],
        'policy_means': [],
        'policy_stds': [],
        'advantages_by_stage': {},  # store advantages at different stages
    }
    
    # Stages to snapshot advantage distributions
    snapshot_iters = [0, 50, 100, 199]
    
    for iteration in range(n_iterations):
        # ---- Collect trajectories ----
        data = collect_trajectories(env, policy, value_net, steps_per_iter, gamma, lam)
        
        if data['episode_rewards']:
            history['rewards'].extend(data['episode_rewards'])
        
        # Snapshot advantages at selected iterations
        if iteration in snapshot_iters:
            history['advantages_by_stage'][iteration] = data['advantages'].cpu().numpy().copy()
        
        # ---- PPO update ----
        stats = ppo_update(
            policy, value_net, policy_optimizer, value_optimizer,
            data, clip_eps=clip_eps, n_epochs=n_epochs, batch_size=batch_size
        )
        
        # Aggregate stats
        history['kl_divs'].append(np.mean(stats['kl_divs']))
        history['clip_fractions'].append(np.mean(stats['clip_fractions']))
        history['all_ratios'].extend(stats['ratios'])
        history['policy_losses'].append(np.mean(stats['policy_losses']))
        history['value_losses'].append(np.mean(stats['value_losses']))
        history['entropies'].append(np.mean(stats['entropies']))
        
        # ---- Track policy statistics ----
        with torch.no_grad():
            test_state = torch.zeros(1, state_dim).to(device)
            mu, std = policy.get_mean_std(test_state)
            history['policy_means'].append(mu.cpu().numpy().flatten()[0])
            history['policy_stds'].append(std.cpu().numpy().flatten()[0])
        
        if (iteration + 1) % 50 == 0:
            recent = history['rewards'][-20:] if len(history['rewards']) >= 20 else history['rewards']
            print(f"  PPO iter {iteration+1}/{n_iterations} | "
                  f"Avg Reward: {np.mean(recent):.1f} | "
                  f"KL: {history['kl_divs'][-1]:.4f} | "
                  f"Clip frac: {history['clip_fractions'][-1]:.3f}")
    
    env.close()
    return history


print("Training PPO on Pendulum-v1...")
ppo_history = train_ppo()
print(f"PPO final avg reward (last 20): {np.mean(ppo_history['rewards'][-20:]):.1f}")

---
## 14. Vanilla REINFORCE Training (for Comparison)

In [ ]:
# ============================================================
#  Train REINFORCE for comparison
# ============================================================

print("Training vanilla REINFORCE on Pendulum-v1 (for comparison)...")
reinforce_history = train_reinforce()
print(f"REINFORCE final avg reward (last 20): {np.mean(reinforce_history['rewards'][-20:]):.1f}")

---
## 15. Visualization 1 -- Surrogate Objective Landscape

In [ ]:
# ============================================================
#  Visualization 1: Surrogate Objective Landscape
#  Shows L(r) and clipped L(r) as a function of the ratio r
#  for both positive and negative advantages.
# ============================================================

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
r = np.linspace(0.0, 2.5, 500)
eps = CLIP_EPS

for idx, (A_val, title) in enumerate([(1.0, 'Positive Advantage ($\\hat{A} > 0$)'),
                                       (-1.0, 'Negative Advantage ($\\hat{A} < 0$)')]):
    ax = axes[idx]
    
    # Unclipped surrogate: r * A
    L_unclipped = r * A_val
    
    # Clipped surrogate: clip(r, 1-eps, 1+eps) * A
    r_clipped = np.clip(r, 1 - eps, 1 + eps)
    L_clipped = r_clipped * A_val
    
    # PPO objective: min(unclipped, clipped)
    L_ppo = np.minimum(L_unclipped, L_clipped)
    
    ax.plot(r, L_unclipped, color=COLORS['trpo'], linestyle='--', alpha=0.7,
            label='$r \\cdot \\hat{A}$ (unclipped)')
    ax.plot(r, L_clipped, color=COLORS['highlight'], linestyle='--', alpha=0.7,
            label='$\\mathrm{clip}(r) \\cdot \\hat{A}$')
    ax.plot(r, L_ppo, color=COLORS['ppo'], linewidth=3,
            label='$L^{CLIP}$ (PPO objective)')
    
    # Mark clipping boundaries
    ax.axvline(x=1 - eps, color='gray', linestyle=':', alpha=0.5)
    ax.axvline(x=1 + eps, color='gray', linestyle=':', alpha=0.5)
    ax.axvline(x=1.0, color='gray', linestyle='-', alpha=0.3)
    
    # Shade the clipping region
    ax.axvspan(1 - eps, 1 + eps, alpha=0.08, color='gray')
    
    ax.set_xlabel('Importance Ratio $r_t(\\theta)$')
    ax.set_ylabel('Objective')
    ax.set_title(title)
    ax.legend(loc='best', fontsize=10)
    ax.set_xlim(0, 2.5)

plt.suptitle('PPO Clipped Surrogate Objective Landscape', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 16. Visualization 2 -- Training Reward Curves: TRPO vs PPO

In [ ]:
# ============================================================
#  Visualization 2: Training Reward Curves (TRPO vs PPO vs REINFORCE)
# ============================================================

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
window = 20

# ---- Raw rewards ----
ax = axes[0]
ax.plot(trpo_history['rewards'], alpha=0.15, color=COLORS['trpo'])
ax.plot(ppo_history['rewards'], alpha=0.15, color=COLORS['ppo'])
ax.plot(reinforce_history['rewards'], alpha=0.15, color=COLORS['reinforce'])
ax.set_title('Raw Episode Rewards')
ax.set_xlabel('Episode')
ax.set_ylabel('Total Reward')
ax.legend(['TRPO', 'PPO', 'REINFORCE'], loc='lower right')

# ---- Smoothed rewards ----
ax = axes[1]
for rewards, color, label in [
    (trpo_history['rewards'], COLORS['trpo'], 'TRPO'),
    (ppo_history['rewards'], COLORS['ppo'], 'PPO'),
    (reinforce_history['rewards'], COLORS['reinforce'], 'REINFORCE'),
]:
    if len(rewards) >= window:
        s = smooth(rewards, window)
        ax.plot(s, color=color, label=label)

ax.axhline(y=-300, color='gray', linestyle='--', alpha=0.7, label='Target (-300)')
ax.set_title(f'Smoothed Rewards (window={window})')
ax.set_xlabel('Episode')
ax.set_ylabel('Avg Reward')
ax.legend(loc='lower right')

plt.suptitle('Pendulum-v1: TRPO vs PPO vs REINFORCE', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 17. Visualization 3 -- KL Divergence Over Training

In [ ]:
# ============================================================
#  Visualization 3: KL Divergence Over Training
# ============================================================

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# ---- TRPO KL ----
ax = axes[0]
ax.plot(trpo_history['kl_divs'], color=COLORS['trpo'], alpha=0.7)
if len(trpo_history['kl_divs']) >= 10:
    ax.plot(smooth(trpo_history['kl_divs'], 10), color=COLORS['trpo'], linewidth=2.5)
ax.axhline(y=TRPO_DELTA, color='red', linestyle='--', alpha=0.7,
           label=f'Target $\\delta$ = {TRPO_DELTA}')
ax.set_title('TRPO: KL Divergence per Update')
ax.set_xlabel('Iteration')
ax.set_ylabel('$D_{KL}(\\pi_{old} \\| \\pi_{new})$')
ax.legend()

# ---- PPO KL ----
ax = axes[1]
ax.plot(ppo_history['kl_divs'], color=COLORS['ppo'], alpha=0.7)
if len(ppo_history['kl_divs']) >= 10:
    ax.plot(smooth(ppo_history['kl_divs'], 10), color=COLORS['ppo'], linewidth=2.5)
ax.axhline(y=KL_TARGET, color='red', linestyle='--', alpha=0.7,
           label=f'Reference KL = {KL_TARGET}')
ax.set_title('PPO: KL Divergence per Update')
ax.set_xlabel('Iteration')
ax.set_ylabel('$D_{KL}(\\pi_{old} \\| \\pi_{new})$')
ax.legend()

# ---- REINFORCE KL ----
ax = axes[2]
ax.plot(reinforce_history['kl_divs'], color=COLORS['reinforce'], alpha=0.7)
if len(reinforce_history['kl_divs']) >= 10:
    ax.plot(smooth(reinforce_history['kl_divs'], 10), color=COLORS['reinforce'], linewidth=2.5)
ax.axhline(y=KL_TARGET, color='red', linestyle='--', alpha=0.7,
           label=f'Reference KL = {KL_TARGET}')
ax.set_title('REINFORCE: KL Divergence per Update')
ax.set_xlabel('Iteration')
ax.set_ylabel('$D_{KL}(\\pi_{old} \\| \\pi_{new})$')
ax.legend()

plt.suptitle('KL Divergence Tracking: Trust Region Methods vs Unconstrained',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 18. Visualization 4 -- PPO Clipping Behavior

In [ ]:
# ============================================================
#  Visualization 4: PPO Clipping Behavior
#  Histogram of importance ratios + clip fraction over training
# ============================================================

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ---- Histogram of all importance ratios ----
ax = axes[0]
ratios_arr = np.array(ppo_history['all_ratios'])
# Clip for visualization (some ratios may be extreme)
ratios_vis = ratios_arr[(ratios_arr > 0) & (ratios_arr < 3.0)]
ax.hist(ratios_vis, bins=100, color=COLORS['ppo'], alpha=0.7, edgecolor='white',
        density=True)
ax.axvline(x=1.0, color='black', linestyle='-', linewidth=1.5, label='$r = 1$')
ax.axvline(x=1 - CLIP_EPS, color='red', linestyle='--', linewidth=1.5,
           label=f'$1 - \\epsilon = {1-CLIP_EPS}$')
ax.axvline(x=1 + CLIP_EPS, color='red', linestyle='--', linewidth=1.5,
           label=f'$1 + \\epsilon = {1+CLIP_EPS}$')
ax.set_xlabel('Importance Ratio $r_t(\\theta)$')
ax.set_ylabel('Density')
ax.set_title('Distribution of Importance Ratios (All Updates)')
ax.legend(fontsize=10)

# ---- Clip fraction over training ----
ax = axes[1]
ax.plot(ppo_history['clip_fractions'], color=COLORS['ppo'], alpha=0.5)
if len(ppo_history['clip_fractions']) >= 10:
    ax.plot(smooth(ppo_history['clip_fractions'], 10), color=COLORS['ppo'],
            linewidth=2.5, label='Smoothed')
ax.axhline(y=0.05, color='gray', linestyle=':', alpha=0.7, label='Lower ref (0.05)')
ax.axhline(y=0.3, color='gray', linestyle=':', alpha=0.7, label='Upper ref (0.3)')
ax.set_xlabel('Iteration')
ax.set_ylabel('Clip Fraction')
ax.set_title('PPO Clip Fraction Over Training')
ax.legend()

plt.suptitle('PPO Clipping Analysis', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 19. Visualization 5 -- Policy Mean and Std Evolution

In [ ]:
# ============================================================
#  Visualization 5: Policy Mean/Std Evolution Over Training
# ============================================================

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# ---- TRPO policy mean ----
ax = axes[0, 0]
ax.plot(trpo_history['policy_means'], color=COLORS['trpo'])
ax.set_title('TRPO: Policy Mean $\\mu$ at Origin State')
ax.set_xlabel('Iteration')
ax.set_ylabel('Mean Action')

# ---- TRPO policy std ----
ax = axes[0, 1]
ax.plot(trpo_history['policy_stds'], color=COLORS['trpo'])
ax.set_title('TRPO: Policy Std $\\sigma$ (Exploration)')
ax.set_xlabel('Iteration')
ax.set_ylabel('Standard Deviation')

# ---- PPO policy mean ----
ax = axes[1, 0]
ax.plot(ppo_history['policy_means'], color=COLORS['ppo'])
ax.set_title('PPO: Policy Mean $\\mu$ at Origin State')
ax.set_xlabel('Iteration')
ax.set_ylabel('Mean Action')

# ---- PPO policy std ----
ax = axes[1, 1]
ax.plot(ppo_history['policy_stds'], color=COLORS['ppo'])
ax.set_title('PPO: Policy Std $\\sigma$ (Exploration)')
ax.set_xlabel('Iteration')
ax.set_ylabel('Standard Deviation')

plt.suptitle('Policy Parameter Evolution During Training', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 20. Visualization 6 -- Advantage Distribution at Different Training Stages

In [ ]:
# ============================================================
#  Visualization 6: Advantage Distribution at Different Stages
# ============================================================

snapshot_iters = sorted(ppo_history['advantages_by_stage'].keys())
n_snaps = len(snapshot_iters)

fig, axes = plt.subplots(1, n_snaps, figsize=(4 * n_snaps, 5))
if n_snaps == 1:
    axes = [axes]

colors_snap = [COLORS['trpo'], COLORS['ppo'], COLORS['reinforce'], COLORS['highlight']]

for i, it in enumerate(snapshot_iters):
    ax = axes[i]
    advs = ppo_history['advantages_by_stage'][it]
    color = colors_snap[i % len(colors_snap)]
    
    ax.hist(advs, bins=50, color=color, alpha=0.7, edgecolor='white', density=True)
    ax.axvline(x=0, color='black', linestyle='--', alpha=0.5)
    ax.set_title(f'Iteration {it}', fontsize=12)
    ax.set_xlabel('Advantage $\\hat{A}_t$')
    ax.set_ylabel('Density')
    
    # Annotate with statistics
    ax.text(0.02, 0.95, f'$\\mu$ = {advs.mean():.3f}\n$\\sigma$ = {advs.std():.3f}',
            transform=ax.transAxes, fontsize=10, verticalalignment='top',
            bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))

plt.suptitle('PPO: Advantage Distribution Evolution', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 21. Verification

In [ ]:
# ============================================================
#  Verification: Automated Checks
# ============================================================

print("=" * 60)
print("  VERIFICATION RESULTS")
print("=" * 60)

# ---- Check 1: TRPO improves reward monotonically (mostly) ----
# We check that the smoothed reward trend is non-decreasing for at least 70% of the steps
trpo_smoothed = smooth(trpo_history['rewards'], 20)
if len(trpo_smoothed) > 1:
    diffs = np.diff(trpo_smoothed)
    frac_increasing = (diffs >= 0).mean()
    # Also check that final reward is better than initial
    improved = trpo_smoothed[-1] > trpo_smoothed[0]
    trpo_monotonic = improved
else:
    trpo_monotonic = False
    frac_increasing = 0.0

tag1 = "[PASS]" if trpo_monotonic else "[FAIL]"
print(f"\n{tag1} TRPO improves reward over training")
print(f"       Initial smoothed: {trpo_smoothed[0]:.1f} -> "
      f"Final smoothed: {trpo_smoothed[-1]:.1f} | "
      f"Fraction increasing: {frac_increasing:.1%}")

# ---- Check 2: PPO solves Pendulum (avg reward > -300) ----
ppo_final_avg = np.mean(ppo_history['rewards'][-20:])
ppo_solved = ppo_final_avg > -300
tag2 = "[PASS]" if ppo_solved else "[FAIL]"
print(f"\n{tag2} PPO solves Pendulum (avg reward > -300)")
print(f"       Final avg reward (last 20 episodes): {ppo_final_avg:.1f}")

# ---- Check 3: KL divergence stays near target for TRPO ----
# Filter out zero KL values (steps not taken)
nonzero_kls = [kl for kl in trpo_history['kl_divs'] if kl > 0]
if nonzero_kls:
    avg_trpo_kl = np.mean(nonzero_kls)
    kl_near_target = avg_trpo_kl <= TRPO_DELTA * 3  # within 3x of target
else:
    avg_trpo_kl = 0.0
    kl_near_target = False

tag3 = "[PASS]" if kl_near_target else "[FAIL]"
print(f"\n{tag3} KL divergence stays near target for TRPO")
print(f"       Avg KL (nonzero steps): {avg_trpo_kl:.5f} | "
      f"Target delta: {TRPO_DELTA}")

# ---- Check 4: PPO clip fraction is reasonable (between 0.05 and 0.3) ----
avg_clip_frac = np.mean(ppo_history['clip_fractions'])
clip_reasonable = 0.01 <= avg_clip_frac <= 0.5
tag4 = "[PASS]" if clip_reasonable else "[FAIL]"
print(f"\n{tag4} PPO clip fraction is reasonable")
print(f"       Avg clip fraction: {avg_clip_frac:.3f} (expected 0.05-0.30)")

# ---- Check 5: Both methods outperform vanilla REINFORCE ----
reinforce_final_avg = np.mean(reinforce_history['rewards'][-20:])
trpo_final_avg = np.mean(trpo_history['rewards'][-20:])
trpo_better = trpo_final_avg > reinforce_final_avg
ppo_better = ppo_final_avg > reinforce_final_avg
both_better = trpo_better or ppo_better  # at least one outperforms
tag5 = "[PASS]" if both_better else "[FAIL]"
print(f"\n{tag5} Trust region methods outperform vanilla REINFORCE")
print(f"       TRPO: {trpo_final_avg:.1f} | PPO: {ppo_final_avg:.1f} | "
      f"REINFORCE: {reinforce_final_avg:.1f}")

print("\n" + "=" * 60)
n_pass = sum([trpo_monotonic, ppo_solved, kl_near_target, clip_reasonable, both_better])
print(f"  {n_pass}/5 checks passed")
print("=" * 60)

---
## 22. Summary and Key Takeaways

### What We Covered

1. **The Policy Update Problem** -- Standard policy gradients suffer from instability because large updates can catastrophically degrade performance. The data distribution depends on the policy, creating a destructive feedback loop when the policy changes too much.

2. **Surrogate Objective** -- Using importance sampling, we can evaluate a new policy $\pi_\theta$ using data collected under the old policy $\pi_{\theta_{\text{old}}}$. The ratio $r_t(\theta) = \pi_\theta / \pi_{\theta_{\text{old}}}$ reweights old experiences.

3. **TRPO** -- Constrains the update with $D_{KL}(\pi_{old} \| \pi_{new}) \leq \delta$ to guarantee monotonic improvement. Uses conjugate gradient to compute the natural gradient direction $F^{-1}g$ and backtracking line search for the step size.

4. **PPO** -- Achieves similar stability by clipping the importance ratio: $\text{clip}(r_t, 1 - \epsilon, 1 + \epsilon)$. Much simpler to implement (no CG, no FVP, no line search) while maintaining comparable performance.

5. **KL Divergence Tracking** -- TRPO directly constrains KL, keeping updates conservative. PPO implicitly controls it through clipping. Both yield more stable training than unconstrained methods.

### Key Observations

- **TRPO** provides principled, theoretically-grounded updates but is complex to implement and computationally expensive (conjugate gradient, line search).
- **PPO** is the practical workhorse: simple, efficient, and almost always "good enough." It is the default algorithm at OpenAI and in many RL libraries.
- **KL divergence** is the key metric for training stability. Methods that control it (explicitly or implicitly) consistently outperform unconstrained approaches.
- **GAE** provides a crucial variance reduction mechanism that both TRPO and PPO rely on.
- **Policy standard deviation** naturally decreases during training as the agent becomes more confident, reflecting a transition from exploration to exploitation.

### Looking Forward

- **PPO variants**: Adaptive KL penalty (PPO-KL), dual-clip for offline RL, multi-agent PPO (MAPPO)
- **SAC (Soft Actor-Critic)**: Entropy-regularized off-policy method for continuous control
- **RLHF**: PPO is the dominant algorithm for reinforcement learning from human feedback in LLM alignment
- **Model-based + PPO**: Dreamer, MBPO combine learned models with PPO-style updates